# 📘 学习注释版：Silver CRM Customer

**Input：** `workspace.bronze.crm_cust_info`  
**Output：** `workspace.silver.crm_customers`

主要学习：
- Trim 字符串
- 婚姻状态/性别标准化
- 删除缺失 Customer ID
- 字段统一重命名
- DataFrame → Delta Table


#Initialization

## 🧰 学习说明：导入 PySpark 函数/类型

这里仅准备后续清洗需要的 API，例如：
- `col()`：引用列
- `trim()`：去前后空格
- `StringType`：判断字符串类型
- `F.when()`：类似 SQL `CASE WHEN`

这一 Cell 不改变数据。


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

# Read Bronze table

## 📖 学习说明：读取 Bronze Table

**Input：** `workspace.bronze.crm_cust_info`  
**Process：** `spark.table()` 把 Catalog 中的 Table 读取成 DataFrame  
**Output：** 变量 `df`

后续所有 Silver 清洗都在这个 DataFrame 上进行。


In [0]:
df = spark.table("workspace.bronze.crm_cust_info")

#Silver Transformations

##Trimming

## ✂️ 学习说明：批量 Trim 字符串

遍历 DataFrame 所有字段：

```text
如果字段类型 = String
→ trim()
→ 去掉前后空格
```

真实数据中 `" Jon"` 和 `"Jon"` 在比较/JOIN 时可能被认为不同，所以 Silver 要先清理。


In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

##Normalization

## 🔄 学习说明：客户代码值标准化

把源系统中的短 Code 变成统一业务值，例如：

```text
S → Single
M → Married
其他 → n/a

F → Female
M → Male
其他 → n/a
```

这就是 Silver 的典型职责：**源系统表示方式 → 统一业务表示方式**。


In [0]:

df = (
    df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
         .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
         .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "F", "Female")
         .when(F.upper(F.col("cst_gndr")) == "M", "Male")
         .otherwise("n/a")
    )
)

##Remove Records with Missing Customer ID

## 🧹 学习说明：删除没有 Customer ID 的记录

`customer_id` 是客户数据的重要业务标识。

```text
cst_id IS NULL
→ 无法可靠识别客户
→ Silver 中过滤掉
```

对应 SQL 思维：`WHERE cst_id IS NOT NULL`。


In [0]:
df = df.filter(col("cst_id").isNotNull())

## Renaming Columns

## 🔄 学习说明：客户代码值标准化

把源系统中的短 Code 变成统一业务值，例如：

```text
S → Single
M → Married
其他 → n/a

F → Female
M → Male
其他 → n/a
```

这就是 Silver 的典型职责：**源系统表示方式 → 统一业务表示方式**。


In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Sanity checks of dataframe

## 👀 学习说明：DataFrame Sanity Check

只显示前 10 行，快速确认当前 DataFrame：
- 字段是否正确
- 清洗是否生效
- 数据是否仍然存在

这一步不写表，只是开发时的中间检查。


In [0]:
df.limit(10).display()

#Writing Silver Table

## 💾 学习说明：把 DataFrame 持久化为 Delta Table

**Input：** 当前 `df`  
**Process：**
- `mode("overwrite")`：目标已存在时覆盖
- `format("delta")`：使用 Delta 格式
- `saveAsTable()`：注册为 Catalog Table

**Output：** `workspace.silver.crm_customers`

注意：这也是为什么 Bootcamp 可以重复运行而通常不会因为“表已存在”直接失败。


In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_customers")

## Sanity checks of silver table

## ✅ 学习说明：验证 Silver Customer

这是 **Sanity Check（合理性检查）**。

目的不是加工数据，而是确认刚写入的 Silver Table：
- 能正常查询
- 字段名正确
- 清洗结果基本符合预期


### 💡 VS Code 阅读版：Databricks SQL（仅展示，不在本地执行）

```sql
SELECT * FROM workspace.silver.crm_customers LIMIT 10
```

> 原可执行版本仍保留在 `01_可执行注释版`。在 Databricks 中请执行那一版。
